In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_path = "runs/default/2-3/final/"
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load tokenizer and model
#tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path).to(device)

import torch

# --- 1. Get input (already mapped vocab) ---
input_ids = ds[10]["input_ids"][:13]

# ensure tensor is 1 x seq_len
input_ids = input_ids.unsqueeze(0).to(device)

# --- 2. Attention mask (needed for HF generate) ---
attention_mask = torch.ones_like(input_ids).to(device)

# --- 3. Generate in MODEL (new vocab space) ---
outputs = model.generate(
    input_ids=input_ids,
    attention_mask=attention_mask,
    max_length=50,
    pad_token_id=tokenizer.eos_token_id  # avoids warning
)

# --- 4. Map outputs BACK to original tokenizer vocab ---
remapped_outputs = torch.tensor(
    [
        [int(new_to_old[int(tok)]) for tok in seq]
        for seq in outputs
    ],
    device=outputs.device
)

# --- 5. Decode with original tokenizer ---
print(
    tokenizer.decode(remapped_outputs[0], skip_special_tokens=True)
)


device = next(model.parameters()).device

# --------------------------------------------------
# 1. INPUT: raw script
# --------------------------------------------------
text = " Ava Hailey Johnson was born"

# --------------------------------------------------
# 2. Tokenize with ORIGINAL tokenizer
# --------------------------------------------------
old_input_ids = tokenizer.encode(text, return_tensors=None)  # list[int]

# --------------------------------------------------
# 3. Map OLD vocab → NEW vocab (STRICT, no fallback)
# --------------------------------------------------
new_input_ids = torch.tensor(
    [old_to_new[int(tok)] for tok in old_input_ids],
    dtype=torch.long
).unsqueeze(0).to(device)

# --------------------------------------------------
# 4. Attention mask
# --------------------------------------------------
attention_mask = torch.ones_like(new_input_ids).to(device)

# --------------------------------------------------
# 5. Generate in reduced vocab model
# --------------------------------------------------
outputs = model.generate(
    input_ids=new_input_ids,
    attention_mask=attention_mask,
    max_length=50,
    pad_token_id=tokenizer.eos_token_id
)

# --------------------------------------------------
# 6. Map outputs BACK (STRICT inverse mapping)
# --------------------------------------------------
outputs_cpu = outputs.cpu()

remapped_outputs = torch.tensor(
    [
        [int(new_to_old[int(tok)]) for tok in seq]
        for seq in outputs_cpu
    ],
    device=device
)

# --------------------------------------------------
# 7. Decode
# --------------------------------------------------
print(tokenizer.decode(remapped_outputs[0], skip_special_tokens=True))